In [6]:
import torch
import torch.nn as nn
import numpy as np
import networkx as nx
from tqdm import tqdm
from collections import defaultdict
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split


In [8]:
from os import chdir
from pathlib import Path

cwd = Path.cwd()
print(f"CWD: {cwd}")
if cwd.name == "code":
    chdir("..")
print(f"CWD: {Path.cwd()}")

CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main
CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main


In [10]:
class UserModel(nn.Module):
    def __init__(self, num_items, latent_dim):
        super().__init__()
        self.item_emb = nn.Embedding(num_items, latent_dim)
        nn.init.normal_(self.item_emb.weight, std=0.1)

    def forward(self, item_ids):
        return self.item_emb(item_ids)


In [18]:
def build_user_graph(interactions, num_users, k=5):
    G = nx.Graph()
    for u in range(num_users): G.add_node(u)
    user_item = defaultdict(set)
    for u, i, _ in interactions:
        user_item[u].add(i)
    for u in range(num_users):
        sims = []
        for v in range(num_users):
            if u == v: continue
            sim = len(user_item[u] & user_item[v])
            sims.append((v, sim))
        top_k = sorted(sims, key=lambda x: -x[1])[:k]
        for v, _ in top_k:
            G.add_edge(u, v)
    return G

def gossip_learning(interactions, num_users, num_items, latent_dim=10, lr=0.01, epochs=20, k=5):
    models = {u: UserModel(num_items, latent_dim) for u in range(num_users)}
    optims = {u: torch.optim.SGD(models[u].parameters(), lr=lr) for u in range(num_users)}
    criterion = nn.MSELoss()
    
    graph = build_user_graph(interactions, num_users, k)
    data_by_user = defaultdict(list)
    for u, i, r in interactions:
        data_by_user[u].append((i, r))
    
    for epoch in range(epochs):
        total_loss = 0.0
        total_count = 0
        for u in range(num_users):
            model = models[u]
            optimizer = optims[u]
            model.train()
            optimizer.zero_grad()
            
            items, ratings = zip(*data_by_user[u]) if data_by_user[u] else ([], [])
            if not items: continue
            
            item_ids = torch.LongTensor(items)
            rating_vals = torch.FloatTensor(ratings)

            pred = model(item_ids).sum(dim=1)
            loss = criterion(pred, rating_vals)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            total_count += len(ratings)
        
        # Gossip step: share embeddings with neighbors
        with torch.no_grad():
            new_weights = {}
            for u in range(num_users):
                neighbors = list(graph.neighbors(u)) + [u]
                emb_sum = sum([models[v].item_emb.weight.data for v in neighbors])
                new_weights[u] = emb_sum / len(neighbors)
            for u in range(num_users):
                models[u].item_emb.weight.data = new_weights[u].clone()

        print(f"Epoch {epoch+1}: Loss={total_loss/total_count:.4f}, Communication={num_users * k} messages")

    return models

def evaluate(models, test_data):
    y_true, y_pred = [], []
    for u, i, r in test_data:
        if u not in models: continue
        model = models[u]
        model.eval()
        with torch.no_grad():
            pred = model(torch.LongTensor([i])).sum().item()
        y_true.append(r)
        y_pred.append(pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"Test RMSE: {rmse:.4f}")
    return rmse


In [22]:
def gossip_learning_tuned(
    interactions, num_users, num_items,
    latent_dim=32, lr=0.005, weight_decay=1e-4,
    epochs=50, k=10
):
    models = {u: UserModel(num_items, latent_dim) for u in range(num_users)}
    optims = {
        u: torch.optim.SGD(models[u].parameters(), lr=lr, weight_decay=weight_decay)
        for u in range(num_users)
    }
    criterion = nn.MSELoss()
    
    graph = build_user_graph(interactions, num_users, k)
    data_by_user = defaultdict(list)
    for u, i, r in interactions:
        data_by_user[u].append((i, r))
    
    for epoch in range(epochs):
        total_loss = 0.0
        total_count = 0
        for u in range(num_users):
            model = models[u]
            optimizer = optims[u]
            model.train()
            optimizer.zero_grad()
            
            items, ratings = zip(*data_by_user[u]) if data_by_user[u] else ([], [])
            if not items: continue
            
            item_ids = torch.LongTensor(items)
            rating_vals = torch.FloatTensor(ratings)

            pred = model(item_ids).sum(dim=1)
            loss = criterion(pred, rating_vals)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            total_count += len(ratings)
        
        with torch.no_grad():
            new_weights = {}
            for u in range(num_users):
                neighbors = list(graph.neighbors(u)) + [u]
                emb_sum = sum([models[v].item_emb.weight.data for v in neighbors])
                new_weights[u] = emb_sum / len(neighbors)
            for u in range(num_users):
                models[u].item_emb.weight.data = new_weights[u].clone()

        print(f"Epoch {epoch+1}: Loss={total_loss/total_count:.4f}, Communication={num_users * k} messages")

    return models

In [20]:
data = np.loadtxt("dataset/u.data", dtype=int)
data[:, :2] -= 1  # shift user/item id to zero-indexed
train, test = train_test_split(data, test_size=0.25, random_state=10)
train = train[:, :3]
test = test[:, :3]
models = gossip_learning(train, num_users=943, num_items=1682)
evaluate(models, test)

Epoch 1: Loss=0.1789, Communication=4715 messages
Epoch 2: Loss=0.1773, Communication=4715 messages
Epoch 3: Loss=0.1769, Communication=4715 messages
Epoch 4: Loss=0.1767, Communication=4715 messages
Epoch 5: Loss=0.1765, Communication=4715 messages
Epoch 6: Loss=0.1763, Communication=4715 messages
Epoch 7: Loss=0.1761, Communication=4715 messages
Epoch 8: Loss=0.1760, Communication=4715 messages
Epoch 9: Loss=0.1758, Communication=4715 messages
Epoch 10: Loss=0.1757, Communication=4715 messages
Epoch 11: Loss=0.1755, Communication=4715 messages
Epoch 12: Loss=0.1754, Communication=4715 messages
Epoch 13: Loss=0.1752, Communication=4715 messages
Epoch 14: Loss=0.1751, Communication=4715 messages
Epoch 15: Loss=0.1749, Communication=4715 messages
Epoch 16: Loss=0.1748, Communication=4715 messages
Epoch 17: Loss=0.1747, Communication=4715 messages
Epoch 18: Loss=0.1745, Communication=4715 messages
Epoch 19: Loss=0.1744, Communication=4715 messages
Epoch 20: Loss=0.1743, Communication=471

3.678480750814954

In [24]:
models = gossip_learning_tuned(
    train, num_users=943, num_items=1682,
    latent_dim=32, lr=0.005, weight_decay=1e-4,
    epochs=50, k=10
)
evaluate(models, test)

Epoch 1: Loss=0.1818, Communication=9430 messages
Epoch 2: Loss=0.1777, Communication=9430 messages
Epoch 3: Loss=0.1768, Communication=9430 messages
Epoch 4: Loss=0.1765, Communication=9430 messages
Epoch 5: Loss=0.1762, Communication=9430 messages
Epoch 6: Loss=0.1759, Communication=9430 messages
Epoch 7: Loss=0.1757, Communication=9430 messages
Epoch 8: Loss=0.1754, Communication=9430 messages
Epoch 9: Loss=0.1752, Communication=9430 messages
Epoch 10: Loss=0.1750, Communication=9430 messages
Epoch 11: Loss=0.1747, Communication=9430 messages
Epoch 12: Loss=0.1745, Communication=9430 messages
Epoch 13: Loss=0.1743, Communication=9430 messages
Epoch 14: Loss=0.1741, Communication=9430 messages
Epoch 15: Loss=0.1738, Communication=9430 messages
Epoch 16: Loss=0.1736, Communication=9430 messages
Epoch 17: Loss=0.1734, Communication=9430 messages
Epoch 18: Loss=0.1732, Communication=9430 messages
Epoch 19: Loss=0.1730, Communication=9430 messages
Epoch 20: Loss=0.1728, Communication=943

3.609772722510103